In [2]:
import pandas as pd
import pickle
import openpyxl
import numpy as np

EXCEL_FILE = "formatted_results_9-16_latest.xlsx"

def export_excel_from_checkpoint(checkpoint_file):
    with open(checkpoint_file, 'rb') as f:
        ck = pickle.load(f)

    all_results              = ck['all_results']
    selected_features_per_ds = ck.get('selected_features_per_ds[ds_name]', {})

    model_map = {
        'SVM': 'SVM', 'kNN': 'KNN',
        'Decision Tree': 'DT', 'Random Forest': 'RF', 'MLP': 'MLP'
    }

    def build_sheet(phase):
        all_rows = []
        for i, ds in enumerate(sorted(all_results.keys()), 1):
            block    = all_results[ds][phase]
            sf_folds = selected_features_per_ds.get(ds, [])  # per-fold feature info

            # ── Column headers ──────────────────────────────────────────
            cols = []
            for m in model_map.values():
                cols += [f'{m}-Accuracy', f'{m}-F1']
            cols += [f'{m} Parameters' for m in model_map.values()]

            if phase == 'relieff':
                cols += ['# Features Selected', 'Features Selected']

            all_rows.append([f'Data {i}'] + [''] * len(cols))
            all_rows.append([''] + cols)

            # ── Fold rows ───────────────────────────────────────────────
            n_folds  = len(next(iter(block.values()))['acc'])
            fold_data = []

            for fold in range(n_folds):
                row, num_row = [f'Fold {fold+1}'], []

                for model in model_map:
                    acc = block[model]['acc'][fold]
                    f1  = block[model]['f1'][fold]
                    row     += [round(acc, 4), round(f1, 4)]
                    num_row += [acc, f1]

                for model in model_map:
                    row.append(str(block[model]['params'][fold]))

                # Append feature info columns for ReliefF sheet
                if phase == 'relieff' and fold < len(sf_folds):
                    fold_sf = sf_folds[fold]
                    n_feat  = fold_sf.get('n_features', '')
                    names   = ', '.join(fold_sf.get('feature_names', []))
                    row += [n_feat, names]
                elif phase == 'relieff':
                    row += ['', '']

                all_rows.append(row)
                fold_data.append(num_row)

            # ── Mean ± Std row ──────────────────────────────────────────
            arr      = np.array(fold_data)
            means    = np.mean(arr, axis=0)
            stds     = np.std(arr,  axis=0)
            mean_std = [f'{m:.4f} ± {s:.4f}' for m, s in zip(means, stds)]

            # Summarise feature selection across folds for the mean row
            if phase == 'relieff' and sf_folds:
                avg_n      = np.median([f['n_features'] for f in sf_folds])
                all_names  = [n for f in sf_folds for n in f.get('feature_names', [])]
                freq        = pd.Series(all_names).value_counts()
                common      = ', '.join(freq[freq == n_folds].index.tolist()) or 'None consistent'
                mean_feat   = [f'median={int(avg_n)}', f'Consistent: {common}']
            else:
                mean_feat = ['', ''] if phase == 'relieff' else []

            all_rows.append(['Mean ± Std'] + mean_std + [''] * len(model_map) + mean_feat)
            all_rows.append([''] * (len(cols) + 1))

        return pd.DataFrame(all_rows)

    with pd.ExcelWriter(EXCEL_FILE, engine='openpyxl') as writer:
        build_sheet('baseline').to_excel(
            writer, sheet_name='Baseline', index=False, header=False)
        build_sheet('relieff').to_excel(
            writer, sheet_name='ReliefF',  index=False, header=False)

    print(f"✅  Excel saved → '{EXCEL_FILE}'")

export_excel_from_checkpoint('results_checkpoint_1-8.pkl')

✅  Excel saved → 'formatted_results_9-16_latest.xlsx'
